In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


SEPARAR DATASET EN ACTIVOS / ELIMINADOS

In [ ]:
# Importando la biblioteca pandas para manipulación y análisis de datos
import pandas as pd
business_payments = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/merged_inner.csv')

In [ ]:
business_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21057 entries, 0 to 21056
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          21057 non-null  int64  
 1   cash_request_id             21057 non-null  int64  
 2   type                        21057 non-null  object 
 3   status                      21057 non-null  object 
 4   category                    2196 non-null   object 
 5   total_amount                21057 non-null  float64
 6   reason                      21057 non-null  object 
 7   created_at                  21057 non-null  object 
 8   updated_at                  21057 non-null  object 
 9   paid_at                     15438 non-null  object 
 10  from_date                   6749 non-null   object 
 11  to_date                     6512 non-null   object 
 12  charge_moment               21057 non-null  object 
 13  amount                      210

In [5]:
import pandas as pd

# Cargar el archivo CSV original
file_path = 'drive/MyDrive/ColabNotebooks/Business_Payments/merged_outer.csv'  # Cambia esta ruta según corresponda
data = pd.read_csv(file_path)

# Separar los datos en usuarios activos y eliminados
usuarios_activos = data[data['user_id'].notna()]
usuarios_eliminados = data[data['deleted_account_id'].notna()]

# Guardar los datos en archivos CSV separados
usuarios_activos.to_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos.csv', index=False)
usuarios_eliminados.to_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados.csv', index=False)

print("Archivos generados: 'usuarios_activos.csv' y 'usuarios_eliminados.csv'")


Archivos generados: 'usuarios_activos.csv' y 'usuarios_eliminados.csv'


--------------------------------------------------------------------------------------------------------

INGENIERIA DE *DATOS*



In [3]:
import pandas as pd

In [6]:
usuarios_activos = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos.csv')
usuarios_eliminados = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados.csv')

In [7]:
# Eliminar columnas no necesarias
usuarios_activos = usuarios_activos.drop(columns=['deleted_account_id'], errors='ignore')
usuarios_eliminados = usuarios_eliminados.drop(columns=['user_id'], errors='ignore')

In [8]:
import pandas as pd

# Convertir a datetime si aún no lo es
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], errors="coerce")
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], errors="coerce")

# Verificar si ya tienen zona horaria asignada
if usuarios_activos["created_at"].dt.tz is None:
    timezone_origen = "Europe/Madrid"  # Cambia esto si es otro timezone
    usuarios_activos["created_at"] = usuarios_activos["created_at"].dt.tz_localize(timezone_origen)

if usuarios_eliminados["created_at"].dt.tz is None:
    usuarios_eliminados["created_at"] = usuarios_eliminados["created_at"].dt.tz_localize(timezone_origen)

# Convertir a UTC
usuarios_activos["created_at"] = usuarios_activos["created_at"].dt.tz_convert("UTC")
usuarios_eliminados["created_at"] = usuarios_eliminados["created_at"].dt.tz_convert("UTC")

# Verificar resultado
print(usuarios_activos[["user_id", "created_at"]].head())
print(usuarios_eliminados[["deleted_account_id", "created_at"]].head())


   user_id                       created_at
0    804.0 2019-12-10 19:05:21.596873+00:00
1    231.0 2019-12-10 19:50:12.347780+00:00
2    191.0 2019-12-10 19:13:35.825460+00:00
3    761.0 2019-12-10 19:16:10.880172+00:00
4   7686.0 2020-05-06 09:59:38.877376+00:00
   deleted_account_id                       created_at
0               309.0 2020-02-10 01:11:53.808270+00:00
1              2499.0 2020-06-28 12:06:33.712840+00:00
2               304.0 2020-01-29 13:53:03.343598+00:00
3               304.0 2020-02-05 17:37:56.852948+00:00
4                91.0 2019-12-11 07:30:42.567035+00:00


COLUMNA TOTAL SOLICITUDES

In [10]:
import pandas as pd

# Contar solicitudes por usuario en usuarios activos
solicitudes_activos = usuarios_activos["user_id"].value_counts().reset_index()
solicitudes_activos.columns = ["user_id", "total_solicitudes_usuario"]

# Unir la información al dataset original
usuarios_activos = usuarios_activos.merge(solicitudes_activos, on="user_id", how="left")

# Contar solicitudes por usuario en usuarios eliminados
solicitudes_eliminados = usuarios_eliminados["deleted_account_id"].value_counts().reset_index()
solicitudes_eliminados.columns = ["deleted_account_id", "total_solicitudes_usuario"]

# Unir la información al dataset original
usuarios_eliminados = usuarios_eliminados.merge(solicitudes_eliminados, on="deleted_account_id", how="left")

COLUMNA TOTAL OPERACIONES CANCELADAS O RECHAZADAS

In [11]:
import pandas as pd

# Contar operaciones canceladas o rechazadas en usuarios activos
canceladas_rechazadas_activos = usuarios_activos[
    usuarios_activos["status"].isin(["cancelled", "rejected"])
]["user_id"].value_counts().reset_index()
canceladas_rechazadas_activos.columns = ["user_id", "total_operaciones_canceladas_rechazadas"]

# Unir la información al dataset original
usuarios_activos = usuarios_activos.merge(canceladas_rechazadas_activos, on="user_id", how="left")
usuarios_activos["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)

# Contar operaciones canceladas o rechazadas en usuarios eliminados
canceladas_rechazadas_eliminados = usuarios_eliminados[
    usuarios_eliminados["status"].isin(["cancelled", "rejected"])
]["deleted_account_id"].value_counts().reset_index()
canceladas_rechazadas_eliminados.columns = ["deleted_account_id", "total_operaciones_canceladas_rechazadas"]

# Unir la información al dataset original
usuarios_eliminados = usuarios_eliminados.merge(canceladas_rechazadas_eliminados, on="deleted_account_id", how="left")
usuarios_eliminados["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)

<ipython-input-11-c2487eed4ee4>:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)
<ipython-input-11-c2487eed4ee4>:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].m

COLUMNA MES DE LA SOLICITUD

In [12]:
# Convertir created_at a zona horaria UTC
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True, errors="coerce")
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True, errors="coerce")

# Extraer el mes en formato YYYY-mm
usuarios_activos["mes_solicitud"] = usuarios_activos["created_at"].dt.strftime("%Y-%m")
usuarios_eliminados["mes_solicitud"] = usuarios_eliminados["created_at"].dt.strftime("%Y-%m")

In [13]:
# Convertir created_at a zona horaria UTC
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True, errors="coerce")
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True, errors="coerce")

# Extraer la semana, el nombre del mes y el año
usuarios_activos["semana_solicitud"] = usuarios_activos["created_at"].dt.isocalendar().week.astype(str) + "_" + \
                                       usuarios_activos["created_at"].dt.strftime("%B") + "_" + \
                                       usuarios_activos["created_at"].dt.strftime("%Y")

usuarios_eliminados["semana_solicitud"] = usuarios_eliminados["created_at"].dt.isocalendar().week.astype(str) + "_" + \
                                          usuarios_eliminados["created_at"].dt.strftime("%B") + "_" + \
                                          usuarios_eliminados["created_at"].dt.strftime("%Y")


In [14]:
# Convertir created_at a zona horaria UTC
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True, errors="coerce")
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True, errors="coerce")

# Extraer el día de la semana, la semana del año, el nombre del mes y el año
usuarios_activos["dia_semana_solicitud"] = usuarios_activos["created_at"].dt.strftime("%A") + "_" + \
                                           usuarios_activos["created_at"].dt.isocalendar().week.astype(str) + "_" + \
                                           usuarios_activos["created_at"].dt.strftime("%B") + "_" + \
                                           usuarios_activos["created_at"].dt.strftime("%Y")

usuarios_eliminados["dia_semana_solicitud"] = usuarios_eliminados["created_at"].dt.strftime("%A") + "_" + \
                                              usuarios_eliminados["created_at"].dt.isocalendar().week.astype(str) + "_" + \
                                              usuarios_eliminados["created_at"].dt.strftime("%B") + "_" + \
                                              usuarios_eliminados["created_at"].dt.strftime("%Y")

# Verificar resultado
print(usuarios_activos[["user_id", "created_at", "dia_semana_solicitud"]].head())
print(usuarios_eliminados[["deleted_account_id", "created_at", "dia_semana_solicitud"]].head())


   user_id                       created_at      dia_semana_solicitud
0    804.0 2019-12-10 19:05:21.596873+00:00  Tuesday_50_December_2019
1    231.0 2019-12-10 19:50:12.347780+00:00  Tuesday_50_December_2019
2    191.0 2019-12-10 19:13:35.825460+00:00  Tuesday_50_December_2019
3    761.0 2019-12-10 19:16:10.880172+00:00  Tuesday_50_December_2019
4   7686.0 2020-05-06 09:59:38.877376+00:00     Wednesday_19_May_2020
   deleted_account_id                       created_at  \
0               309.0 2020-02-10 01:11:53.808270+00:00   
1              2499.0 2020-06-28 12:06:33.712840+00:00   
2               304.0 2020-01-29 13:53:03.343598+00:00   
3               304.0 2020-02-05 17:37:56.852948+00:00   
4                91.0 2019-12-11 07:30:42.567035+00:00   

         dia_semana_solicitud  
0      Monday_7_February_2020  
1         Sunday_26_June_2020  
2    Wednesday_5_January_2020  
3   Wednesday_6_February_2020  
4  Wednesday_50_December_2019  


In [15]:
# Convertir created_at a zona horaria UTC
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True, errors="coerce")
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True, errors="coerce")

# Extraer la hora, día de la semana, la semana del año, el nombre del mes y el año
usuarios_activos["hora_solicitud"] = usuarios_activos["created_at"].dt.hour.astype(str) + "_" + \
                                     usuarios_activos["created_at"].dt.strftime("%A") + "_" + \
                                     usuarios_activos["created_at"].dt.isocalendar().week.astype(str) + "_" + \
                                     usuarios_activos["created_at"].dt.strftime("%B") + "_" + \
                                     usuarios_activos["created_at"].dt.strftime("%Y")

usuarios_eliminados["hora_solicitud"] = usuarios_eliminados["created_at"].dt.hour.astype(str) + "_" + \
                                        usuarios_eliminados["created_at"].dt.strftime("%A") + "_" + \
                                        usuarios_eliminados["created_at"].dt.isocalendar().week.astype(str) + "_" + \
                                        usuarios_eliminados["created_at"].dt.strftime("%B") + "_" + \
                                        usuarios_eliminados["created_at"].dt.strftime("%Y")

# Verificar resultado
print(usuarios_activos[["user_id", "created_at", "hora_solicitud"]].head())
print(usuarios_eliminados[["deleted_account_id", "created_at", "hora_solicitud"]].head())


   user_id                       created_at               hora_solicitud
0    804.0 2019-12-10 19:05:21.596873+00:00  19_Tuesday_50_December_2019
1    231.0 2019-12-10 19:50:12.347780+00:00  19_Tuesday_50_December_2019
2    191.0 2019-12-10 19:13:35.825460+00:00  19_Tuesday_50_December_2019
3    761.0 2019-12-10 19:16:10.880172+00:00  19_Tuesday_50_December_2019
4   7686.0 2020-05-06 09:59:38.877376+00:00      9_Wednesday_19_May_2020
   deleted_account_id                       created_at  \
0               309.0 2020-02-10 01:11:53.808270+00:00   
1              2499.0 2020-06-28 12:06:33.712840+00:00   
2               304.0 2020-01-29 13:53:03.343598+00:00   
3               304.0 2020-02-05 17:37:56.852948+00:00   
4                91.0 2019-12-11 07:30:42.567035+00:00   

                 hora_solicitud  
0      1_Monday_7_February_2020  
1        12_Sunday_26_June_2020  
2   13_Wednesday_5_January_2020  
3  17_Wednesday_6_February_2020  
4  7_Wednesday_50_December_2019  


In [16]:
# Evitar división por cero
usuarios_activos["tasa_rechazo"] = usuarios_activos["total_operaciones_canceladas_rechazadas"] / usuarios_activos["total_solicitudes_usuario"]
usuarios_eliminados["tasa_rechazo"] = usuarios_eliminados["total_operaciones_canceladas_rechazadas"] / usuarios_eliminados["total_solicitudes_usuario"]

# Reemplazar NaN e infinitos por 0
usuarios_activos["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)
usuarios_eliminados["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)

# Verificar resultado
print(usuarios_activos[["user_id", "tasa_rechazo"]].head())
print(usuarios_eliminados[["deleted_account_id", "tasa_rechazo"]].head())


   user_id  tasa_rechazo
0    804.0           1.0
1    231.0           0.2
2    191.0           0.5
3    761.0           1.0
4   7686.0           1.0
   deleted_account_id  tasa_rechazo
0               309.0           1.0
1              2499.0           0.0
2               304.0           1.0
3               304.0           1.0
4                91.0           1.0


<ipython-input-16-702cbbddd062>:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)
<ipython-input-16-702cbbddd062>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col]

In [17]:
# Calcular la diferencia de tiempo entre solicitudes por usuario
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True)
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True)

# Calcular diferencia de días entre solicitudes y sacar el promedio por usuario
intervalo_activos = usuarios_activos.sort_values(["user_id", "created_at"]).groupby("user_id")["created_at"].diff().dt.days
intervalo_eliminados = usuarios_eliminados.sort_values(["deleted_account_id", "created_at"]).groupby("deleted_account_id")["created_at"].diff().dt.days

usuarios_activos["intervalo_promedio_solicitudes"] = usuarios_activos["user_id"].map(intervalo_activos.groupby(usuarios_activos["user_id"]).mean())
usuarios_eliminados["intervalo_promedio_solicitudes"] = usuarios_eliminados["deleted_account_id"].map(intervalo_eliminados.groupby(usuarios_eliminados["deleted_account_id"]).mean())

# Rellenar NaN con un valor alto (por ejemplo, 9999 para usuarios con una sola solicitud)
usuarios_activos["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)
usuarios_eliminados["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)

# Verificar resultado
print(usuarios_activos[["user_id", "intervalo_promedio_solicitudes"]].head())
print(usuarios_eliminados[["deleted_account_id", "intervalo_promedio_solicitudes"]].head())


   user_id  intervalo_promedio_solicitudes
0    804.0                     9999.000000
1    231.0                       32.888889
2    191.0                       61.000000
3    761.0                     9999.000000
4   7686.0                     9999.000000
   deleted_account_id  intervalo_promedio_solicitudes
0               309.0                            11.0
1              2499.0                          9999.0
2               304.0                             7.0
3               304.0                             7.0
4                91.0                          9999.0


<ipython-input-17-2f376469ec71>:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)
<ipython-input-17-2f376469ec71>:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

In [18]:
# Asegurar que las fechas sean tipo datetime
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True, errors="coerce")
usuarios_activos["paid_at"] = pd.to_datetime(usuarios_activos["paid_at"], utc=True, errors="coerce")

usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True, errors="coerce")
usuarios_eliminados["paid_at"] = pd.to_datetime(usuarios_eliminados["paid_at"], utc=True, errors="coerce")

# Calcular la diferencia en días entre pago y creación
usuarios_activos["dias_para_pago"] = (usuarios_activos["paid_at"] - usuarios_activos["created_at"]).dt.days
usuarios_eliminados["dias_para_pago"] = (usuarios_eliminados["paid_at"] - usuarios_eliminados["created_at"]).dt.days

# Definir pagos tardíos (más de 30 días)
usuarios_activos["pago_tardio"] = (usuarios_activos["dias_para_pago"] > 30).astype(int)
usuarios_eliminados["pago_tardio"] = (usuarios_eliminados["dias_para_pago"] > 30).astype(int)

# Calcular la proporción de pagos tardíos por usuario
pago_tardio_ratio_activos = usuarios_activos.groupby("user_id")["pago_tardio"].mean().reset_index()
pago_tardio_ratio_eliminados = usuarios_eliminados.groupby("deleted_account_id")["pago_tardio"].mean().reset_index()

# Renombrar columna
pago_tardio_ratio_activos.columns = ["user_id", "pago_tardio_ratio"]
pago_tardio_ratio_eliminados.columns = ["deleted_account_id", "pago_tardio_ratio"]

# Unir al dataset original
usuarios_activos = usuarios_activos.merge(pago_tardio_ratio_activos, on="user_id", how="left")
usuarios_eliminados = usuarios_eliminados.merge(pago_tardio_ratio_eliminados, on="deleted_account_id", how="left")

# Verificar resultado
print(usuarios_activos[["user_id", "pago_tardio_ratio"]].head())
print(usuarios_eliminados[["deleted_account_id", "pago_tardio_ratio"]].head())



   user_id  pago_tardio_ratio
0    804.0                0.0
1    231.0                0.1
2    191.0                0.0
3    761.0                0.0
4   7686.0                0.0
   deleted_account_id  pago_tardio_ratio
0               309.0                0.0
1              2499.0                0.0
2               304.0                0.0
3               304.0                0.0
4                91.0                0.0


In [19]:
# Contar cuántas veces se ha modificado una solicitud
usuarios_activos["solicitudes_modificadas"] = usuarios_activos.groupby("user_id")["updated_at"].transform("count") - 1
usuarios_eliminados["solicitudes_modificadas"] = usuarios_eliminados.groupby("deleted_account_id")["updated_at"].transform("count") - 1

# Evitar valores negativos (en caso de usuarios con una sola solicitud)
usuarios_activos["solicitudes_modificadas"] = usuarios_activos["solicitudes_modificadas"].clip(lower=0)
usuarios_eliminados["solicitudes_modificadas"] = usuarios_eliminados["solicitudes_modificadas"].clip(lower=0)

# Verificar resultado
print(usuarios_activos[["user_id", "solicitudes_modificadas"]].head())
print(usuarios_eliminados[["deleted_account_id", "solicitudes_modificadas"]].head())


   user_id  solicitudes_modificadas
0    804.0                        0
1    231.0                        9
2    191.0                        1
3    761.0                        0
4   7686.0                        0
   deleted_account_id  solicitudes_modificadas
0               309.0                        1
1              2499.0                        0
2               304.0                        1
3               304.0                        1
4                91.0                        0


In [20]:
# Guardar los archivos actualizados (opcional)
usuarios_activos.to_csv("drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos_actualizado.csv", index=False)
usuarios_eliminados.to_csv("drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados_actualizado.csv", index=False)

In [21]:
usuarios_activos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29522 entries, 0 to 29521
Data columns (total 39 columns):
 #   Column                                   Non-Null Count  Dtype              
---  ------                                   --------------  -----              
 0   id                                       29522 non-null  int64              
 1   amount                                   29522 non-null  float64            
 2   status                                   29522 non-null  object             
 3   created_at                               29522 non-null  datetime64[ns, UTC]
 4   updated_at                               29522 non-null  object             
 5   user_id                                  29522 non-null  float64            
 6   moderated_at                             19434 non-null  object             
 7   reimbursement_date                       29522 non-null  object             
 8   cash_request_received_date               22988 non-null  object   